# S02 - Preprocessing medical et equilibrage train/test

Objectif de cette etape:

1. Garder seulement deux partitions: `train` et `test`.
2. Annuler la validation separee.
3. Pretraiter les images pour ameliorer leur qualite visuelle.
4. Gerer le desequilibre des classes avec class weights et oversampling du train.
5. Sauvegarder les CSV propres pour l'entrainement.

Pourquoi annuler `val` ?

Ici on simplifie le pipeline: le modele apprend sur `train`, puis il est evalue sur `test`. La validation separee n'est plus creee dans cette partie.


## 1. Imports et chemins


In [ ]:
import os
import random
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageOps, ImageFilter

from sklearn.utils.class_weight import compute_class_weight
from sklearn.utils import resample

plt.style.use("default")
sns.set_theme(style="whitegrid")

PROJECT_DIR = Path(r"C:\Users\user\Desktop\Alzheimer_CV_Project")
DATASET_DIR = PROJECT_DIR / "data" / "alzheimer"
TRAIN_DIR = DATASET_DIR / "train"
TEST_DIR = DATASET_DIR / "test"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

print("Dataset:", DATASET_DIR)
print("Train dir exists:", TRAIN_DIR.exists())
print("Test dir exists:", TEST_DIR.exists())
print("Processed dir:", PROCESSED_DIR)


### Interpretation

Cette cellule initialise les chemins du projet. Le dataset contient deja deux dossiers officiels: `train` et `test`. On ne cree plus de dossier `val`.


## 2. Charger les images train/test


In [ ]:
image_extensions = {".jpg", ".jpeg", ".png", ".bmp"}
rows = []

for split_dir in [TRAIN_DIR, TEST_DIR]:
    split_name = split_dir.name
    for class_dir in sorted([p for p in split_dir.iterdir() if p.is_dir()]):
        label = class_dir.name
        for img_path in class_dir.rglob("*"):
            if img_path.suffix.lower() in image_extensions:
                rows.append({
                    "image_path": str(img_path),
                    "split": split_name,
                    "label": label,
                    "file_name": img_path.name,
                })

df = pd.DataFrame(rows)
classes = sorted(df["label"].unique())

print("Nombre total d'images:", len(df))
print("Classes:", classes)
display(df.head())


### Interpretation

Chaque image garde son split original: `train` ou `test`. Le `test` sera conserve pour l'evaluation finale.


## 3. Distribution des classes


In [ ]:
counts = df.groupby(["split", "label"]).size().reset_index(name="n_images")
display(counts)

plt.figure(figsize=(10, 5))
sns.countplot(data=df, x="label", hue="split", order=classes)
plt.title("Distribution des classes train/test")
plt.xlabel("Classe")
plt.ylabel("Nombre d'images")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

class_percent = 100 * df["label"].value_counts(normalize=True).loc[classes]
display(class_percent.rename("percentage"))


### Interpretation

Le desequilibre est clair: `NonDemented` domine fortement, alors que `ModerateDemented` est tres minoritaire. Il faut corriger ce biais avant l'entrainement.


## 4. Verification fuite train/test


In [ ]:
def file_hash(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

df["file_hash"] = df["image_path"].apply(file_hash)
train_hashes = set(df[df["split"] == "train"]["file_hash"])
test_hashes = set(df[df["split"] == "test"]["file_hash"])
common_hashes = train_hashes.intersection(test_hashes)

print("Images train uniques:", len(train_hashes))
print("Images test uniques:", len(test_hashes))
print("Images identiques entre train et test:", len(common_hashes))

if len(common_hashes) > 0:
    leakage_df = df[df["file_hash"].isin(common_hashes)].sort_values("file_hash")
    display(leakage_df.head(30))
else:
    print("Aucune image identique detectee entre train et test.")


### Interpretation

Cette verification evite le data leakage. Si une meme image existe dans train et test, les performances du modele peuvent etre faussement elevees.


## 5. Separer train et test sans validation


In [ ]:
train_df = df[df["split"] == "train"].copy().reset_index(drop=True)
test_df = df[df["split"] == "test"].copy().reset_index(drop=True)

print("Train:", len(train_df))
print("Test:", len(test_df))

display(train_df["label"].value_counts().loc[classes].rename("train_count"))
display(test_df["label"].value_counts().loc[classes].rename("test_count"))


### Interpretation

Il n'y a plus de `val`. On garde uniquement `train` pour apprendre et `test` pour evaluer le modele final.


## 6. Class weights pour les classes minoritaires


In [ ]:
class_weights_values = compute_class_weight(
    class_weight="balanced",
    classes=np.array(classes),
    y=train_df["label"],
)

class_to_index = {class_name: idx for idx, class_name in enumerate(classes)}
class_weights = {class_name: weight for class_name, weight in zip(classes, class_weights_values)}
class_weights_indexed = {class_to_index[class_name]: weight for class_name, weight in class_weights.items()}

class_weights_df = pd.DataFrame({
    "label": list(class_weights.keys()),
    "class_index": [class_to_index[c] for c in class_weights.keys()],
    "class_weight": list(class_weights.values()),
    "train_count": [int((train_df["label"] == c).sum()) for c in class_weights.keys()],
})

display(class_weights_df.sort_values("class_weight", ascending=False))


### Interpretation

Les classes rares recoivent un poids plus eleve. Ainsi, une erreur sur `ModerateDemented` compte plus qu'une erreur sur `NonDemented`.


## 7. Equilibrage du train par oversampling


In [ ]:
max_train_count = train_df["label"].value_counts().max()
balanced_parts = []

for class_name in classes:
    class_part = train_df[train_df["label"] == class_name]
    class_balanced = resample(
        class_part,
        replace=True,
        n_samples=max_train_count,
        random_state=RANDOM_STATE,
    )
    balanced_parts.append(class_balanced)

train_balanced_df = pd.concat(balanced_parts, ignore_index=True)
train_balanced_df = train_balanced_df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
train_balanced_df["split"] = "train_balanced"

print("Train original:", len(train_df))
print("Train balanced:", len(train_balanced_df))
display(train_balanced_df["label"].value_counts().loc[classes])


### Interpretation

Le train equilibre contient le meme nombre d'images par classe. Cela evite que le modele apprenne surtout `NonDemented`.


## 8. Visualisation avant/apres equilibrage


In [ ]:
compare_balance_df = pd.concat([
    train_df.assign(dataset="Train original"),
    train_balanced_df.assign(dataset="Train balanced"),
], ignore_index=True)

plt.figure(figsize=(11, 5))
sns.countplot(data=compare_balance_df, x="label", hue="dataset", order=classes)
plt.title("Equilibrage du train par oversampling")
plt.xlabel("Classe")
plt.ylabel("Nombre d'images")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()


### Interpretation

Le graphique montre que les classes sont maintenant equilibrees dans le train utilise pour l'apprentissage.


## 9. Preprocessing ameliore des images


In [ ]:
def crop_non_black_region(img, threshold=8, padding=8):
    arr = np.array(img)
    mask = arr > threshold
    if mask.sum() == 0:
        return img

    ys, xs = np.where(mask)
    y0, y1 = ys.min(), ys.max()
    x0, x1 = xs.min(), xs.max()

    y0 = max(0, y0 - padding)
    y1 = min(arr.shape[0] - 1, y1 + padding)
    x0 = max(0, x0 - padding)
    x1 = min(arr.shape[1] - 1, x1 + padding)

    return img.crop((x0, y0, x1 + 1, y1 + 1))


def preprocess_image(path, target_size=(224, 224), crop=True, contrast=False, denoise=True):
    img = Image.open(path).convert("L")

    if crop:
        img = crop_non_black_region(img)

    if denoise:
        img = img.filter(ImageFilter.MedianFilter(size=3))

    # Pas d'egalisation de contraste: on garde les intensites plus proches de l'image originale.
    img = img.resize(target_size)
    arr = np.array(img, dtype=np.float32) / 255.0
    return arr

print("Preprocessing ready: crop + denoise + resize + normalization")


### Interpretation

Le preprocessing ameliore les images en plusieurs etapes:

- crop: supprime le fond noir inutile;
- denoise: reduit certains petits bruits;
- resize: standardise la taille;
- normalization: met les pixels entre 0 et 1.

On n'applique pas d'egalisation de contraste, car elle peut modifier artificiellement l'apparence anatomique des IRM.


## 10. Comparaison visuelle du preprocessing


In [ ]:
example_rows = []
for class_name in classes:
    example_rows.append(df[df["label"] == class_name].sample(1, random_state=RANDOM_STATE).iloc[0])

plt.figure(figsize=(14, 4 * len(example_rows)))

for row_idx, row in enumerate(example_rows):
    path = row["image_path"]
    original = Image.open(path).convert("L")
    cropped = crop_non_black_region(original)
    denoised = cropped.filter(ImageFilter.MedianFilter(size=3))
    final_arr = preprocess_image(path, target_size=(224, 224), crop=True, contrast=False, denoise=True)

    visuals = [original, cropped, denoised, final_arr]
    titles = ["Original", "Crop", "Denoise", "Final normalise"]

    for col_idx, (visual, title) in enumerate(zip(visuals, titles)):
        plt.subplot(len(example_rows), 4, row_idx * 4 + col_idx + 1)
        plt.imshow(visual, cmap="gray")
        plt.title(f"{row['label']}\n{title}", fontsize=9)
        plt.axis("off")

plt.suptitle("Preprocessing ameliore par classe", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()


### Interpretation

Cette figure permet de verifier que le preprocessing ameliore la lisibilite sans detruire les structures importantes du cerveau.


## 11. Effet du preprocessing sur les intensites


In [ ]:
intensity_rows = []
sample_df = df.sample(min(500, len(df)), random_state=RANDOM_STATE)

for _, row in sample_df.iterrows():
    original = np.array(Image.open(row["image_path"]).convert("L"), dtype=np.float32) / 255.0
    processed = preprocess_image(row["image_path"], target_size=(224, 224), crop=True, contrast=False, denoise=True)
    intensity_rows.append({
        "label": row["label"],
        "original_mean": original.mean(),
        "processed_mean": processed.mean(),
        "original_std": original.std(),
        "processed_std": processed.std(),
    })

intensity_df = pd.DataFrame(intensity_rows)
display(intensity_df.groupby("label").mean())

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.boxplot(data=intensity_df, x="label", y="original_std", order=classes)
plt.title("Contraste avant preprocessing")
plt.xticks(rotation=25, ha="right")

plt.subplot(1, 2, 2)
sns.boxplot(data=intensity_df, x="label", y="processed_std", order=classes)
plt.title("Contraste apres preprocessing")
plt.xticks(rotation=25, ha="right")

plt.tight_layout()
plt.show()


### Interpretation

`std` represente le contraste global. Apres preprocessing, les structures peuvent devenir plus visibles, ce qui aide le CNN a apprendre des patterns plus stables.


## 12. Data augmentation sur train et test

On ajoute maintenant une augmentation de donnees sur les deux partitions:

- `train_augmented`: utilise pour ameliorer l'apprentissage;
- `test_augmented`: utilise pour tester la robustesse du modele sur des variations d'image.

Important:

Le test original reste la reference principale pour l'evaluation finale. Le test augmente sert comme analyse complementaire de robustesse.


In [ ]:
# ============================================================
# DATA AUGMENTATION TRAIN ET TEST
# ============================================================

from PIL import ImageEnhance

AUGMENTED_DIR = PROCESSED_DIR / "augmented_images"
TRAIN_AUG_DIR = AUGMENTED_DIR / "train_augmented"
TEST_AUG_DIR = AUGMENTED_DIR / "test_augmented"
TRAIN_AUG_DIR.mkdir(parents=True, exist_ok=True)
TEST_AUG_DIR.mkdir(parents=True, exist_ok=True)


def augment_pil_image(img, random_state=None):
    rng = np.random.default_rng(random_state)

    # Transformations legeres et medicalement raisonnables.
    angle = float(rng.uniform(-7, 7))
    translate_x = int(rng.integers(-5, 6))
    translate_y = int(rng.integers(-5, 6))

    aug = img.rotate(angle, resample=Image.BILINEAR, fillcolor=0)
    aug = ImageOps.expand(aug, border=0, fill=0)
    aug = aug.transform(
        aug.size,
        Image.AFFINE,
        (1, 0, translate_x, 0, 1, translate_y),
        resample=Image.BILINEAR,
        fillcolor=0,
    )
    return aug


def create_augmented_dataset(source_df, output_root, copies_per_image=1, prefix="aug"):
    rows = []
    output_root = Path(output_root)

    for row_idx, row in source_df.reset_index(drop=True).iterrows():
        img_path = Path(row["image_path"])
        label = row["label"]
        class_dir = output_root / label
        class_dir.mkdir(parents=True, exist_ok=True)

        original_img = Image.open(img_path).convert("L")
        original_img = crop_non_black_region(original_img)
        original_img = original_img.filter(ImageFilter.MedianFilter(size=3))
        original_img = original_img.resize((224, 224))

        for copy_idx in range(copies_per_image):
            seed = RANDOM_STATE + row_idx * 100 + copy_idx
            aug_img = augment_pil_image(original_img, random_state=seed)
            aug_name = f"{prefix}_{row_idx:05d}_{copy_idx}_{img_path.stem}.png"
            aug_path = class_dir / aug_name
            aug_img.save(aug_path)

            new_row = row.to_dict()
            new_row["image_path"] = str(aug_path)
            new_row["file_name"] = aug_name
            new_row["augmentation"] = prefix
            rows.append(new_row)

    return pd.DataFrame(rows)


# Train augmente: on part du train equilibre pour garder toutes les classes au meme niveau.
train_augmented_df = create_augmented_dataset(
    train_balanced_df,
    TRAIN_AUG_DIR,
    copies_per_image=1,
    prefix="train_aug",
)
train_augmented_df["split"] = "train_augmented"

# Test augmente: utilise uniquement pour evaluation de robustesse, pas comme test principal.
test_augmented_df = create_augmented_dataset(
    test_df,
    TEST_AUG_DIR,
    copies_per_image=1,
    prefix="test_aug",
)
test_augmented_df["split"] = "test_augmented"

print("Train augmented images:", len(train_augmented_df))
print("Test augmented images:", len(test_augmented_df))

display(train_augmented_df["label"].value_counts().loc[classes])
display(test_augmented_df["label"].value_counts().loc[classes])


### Interpretation

L'augmentation applique de petites transformations:

- rotation faible;
- petite translation;
- variations geometriques legeres sans modifier le contraste.

On evite les transformations trop fortes, car une IRM medicale doit rester anatomiquement plausible.

Le train augmente aide le modele a apprendre des variations. Le test augmente permet de verifier si le modele reste stable quand l'image change legerement.


## 13. Visualiser les images augmentees

On compare l'image originale et son augmentation pour quelques exemples de chaque classe.


In [ ]:
# ============================================================
# VISUALISATION ORIGINAL VS AUGMENTE
# ============================================================

plt.figure(figsize=(12, 4 * len(classes)))

for class_idx, class_name in enumerate(classes):
    orig_row = train_balanced_df[train_balanced_df["label"] == class_name].sample(1, random_state=RANDOM_STATE).iloc[0]
    aug_row = train_augmented_df[train_augmented_df["label"] == class_name].sample(1, random_state=RANDOM_STATE).iloc[0]

    orig_img = Image.open(orig_row["image_path"]).convert("L")
    orig_processed = preprocess_image(orig_row["image_path"], target_size=(224, 224), crop=True, contrast=False, denoise=True)
    aug_img = Image.open(aug_row["image_path"]).convert("L")

    plt.subplot(len(classes), 3, class_idx * 3 + 1)
    plt.imshow(orig_img, cmap="gray")
    plt.title(f"{class_name}\nOriginal brut", fontsize=9)
    plt.axis("off")

    plt.subplot(len(classes), 3, class_idx * 3 + 2)
    plt.imshow(orig_processed, cmap="gray")
    plt.title("Original pretraite", fontsize=9)
    plt.axis("off")

    plt.subplot(len(classes), 3, class_idx * 3 + 3)
    plt.imshow(aug_img, cmap="gray")
    plt.title("Augmente", fontsize=9)
    plt.axis("off")

plt.suptitle("Comparaison original / pretraite / augmente", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()


### Interpretation

Les images augmentees doivent rester proches des images originales. Si l'image est trop tournee, deformee ou trop contrastee, l'augmentation devient non medicale.

Ici, les transformations sont volontairement legeres pour conserver la structure du cerveau.


## 14. Sauvegarder les CSV augmentes

On sauvegarde les nouvelles partitions augmentees.


In [ ]:
train_augmented_csv = PROCESSED_DIR / "train_augmented.csv"
test_augmented_csv = PROCESSED_DIR / "test_augmented_robustness.csv"
train_final_csv = PROCESSED_DIR / "train_final_with_augmentation.csv"

# CSV 1: seulement les images augmentees.
train_augmented_df.to_csv(train_augmented_csv, index=False)
test_augmented_df.to_csv(test_augmented_csv, index=False)

# CSV 2: train final = train original equilibre + train augmente.
train_original_for_final = train_balanced_df.copy()
train_original_for_final["augmentation"] = "original_balanced"
train_final_df = pd.concat([
    train_original_for_final,
    train_augmented_df,
], ignore_index=True)
train_final_df = train_final_df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
train_final_df["split"] = "train_final"
train_final_df.to_csv(train_final_csv, index=False)

print("Saved:")
print(train_augmented_csv)
print(test_augmented_csv)
print(train_final_csv)

print("Train balanced original:", len(train_balanced_df))
print("Train augmented only:", len(train_augmented_df))
print("Train final total:", len(train_final_df))

display(train_final_df["label"].value_counts().loc[classes])
display(train_final_df["augmentation"].value_counts())


### Interpretation

Trois fichiers sont sauvegardes:

- `train_augmented.csv`: seulement les images augmentees;
- `test_augmented_robustness.csv`: test augmente pour verifier la robustesse;
- `train_final_with_augmentation.csv`: train final utilise pour le modele.

Le train final contient:

```text
train original equilibre + train augmente
```

Cela permet au modele de voir a la fois les vraies images equilibrees et leurs variantes augmentees.


## 15. Sauvegarde des CSV


In [ ]:
train_csv = PROCESSED_DIR / "train_split.csv"
test_csv = PROCESSED_DIR / "test_split.csv"
train_balanced_csv = PROCESSED_DIR / "train_balanced_oversampled.csv"
class_weights_csv = PROCESSED_DIR / "class_weights.csv"

train_df.to_csv(train_csv, index=False)
test_df.to_csv(test_csv, index=False)
train_balanced_df.to_csv(train_balanced_csv, index=False)
class_weights_df.to_csv(class_weights_csv, index=False)

# Si un ancien val_split existe, on le garde sur disque mais il ne sera plus utilise.
print("Saved:")
print(train_csv)
print(test_csv)
print(train_balanced_csv)
print(class_weights_csv)


### Interpretation

Les splits sauvegardes ne contiennent plus de validation separee. Les prochaines etapes utiliseront `train_balanced_oversampled.csv` et `test_split.csv`.


# Conclusion de l'etape 2

Dans cette version, la validation separee est annulee. Le pipeline utilise seulement:

- `train` pour apprendre;
- `test` pour evaluer.

Le train est equilibre par oversampling afin de reduire le biais vers `NonDemented`.

Le preprocessing ameliore les images avec crop, reduction de bruit, resize et normalisation. Cela permet de donner au CNN des images plus propres et plus comparables.
